<center><h1><strong><font color="blue"> Data Engineering for Data Science - Week 07</font></strong></h1></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Normalization and Case Study</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

**Topics**:
* Normal Forms: 1NF, 2NF, & 3NF
* Denormalization: When and why to break the rules.
* **case Study: in-class Group Discussion**:
   - Normalizations
   - Basic Query
   - Join
   - Aggregations

<center><img alt="" src="images/class-exercise.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Prologue/Motivation ~ Normalization</font></strong></h2></center>

<center><img alt="" src="images/deds-05/motivation-normalization.png" style="height: 600px;"/></center>

> *An **orphan record** problem in an unnormalized database occurs when data that should be logically dependent on another entity becomes disconnected due to the absence of enforced relationships or constraints. For example, a transaction row may reference a customer or product that no longer exists (or was inconsistently modified), because all data resides in a single flat structure without referential integrity. This leads to inconsistencies, invalid references, and unreliable query results, especially during updates or deletions where related data is not systematically maintained*.

<center><h2><strong><font color="blue">Case Study: Global E-Commerce Mini-Store</font></strong></h2></center>

## 1. Overview
This case study examines a simplified **Global Mini-Store**, an online platform where customers from multiple countries purchase technology gadgets. The system currently relies on a **single flat-file structure** (spreadsheet-like table) to record all transactions. While this approach is straightforward and easy to implement at an early stage, it introduces **serious data integrity and scalability issues** as the system grows.

## 2. Problem Context
- Each row represents a **single line item** within a customer order.
- All relevant information—order, customer, product, and transaction—is stored in **one table**.

<center><img alt="" src="images/deds-06/mini-store-case-study.png" style="height: 480px;"/></center>

### Data Components Stored in a Single Table
1. **Order Details**: `order_id`, `order_date`
2. **Customer Information**: `customer_name`, `customer_email`, `customer_country`
3. **Product Details**: `product_id`, `product_name`, `product_category`
4. **Transaction Information**: `unit_price`, `quantity`

<center><h2><strong><font color="blue"> Export-Import Data</font></strong></h2></center>

<center><h3><strong><font color="red"> Please clone (“pull”) the course GitHub repository and locate the file</font><font color="green"> "unnormalized-store-table-example.sql" </font><font color="red"> within the "data" folder</font></strong></h3></center>

### import the SQL sample data to your MySQL server using phpMyadmin as in the previous session.

<center><img alt="" src="images/deds-06/miniStore-initial-table.jpg" style="height: 600px;"/></center>

<center><h3><strong><font color="red"> Pause and carefully examine the table. What are the issues exist in this unstructured (monolithic) table design?</font></strong></h3></center>

<center><img alt="" src="images/stop-exercise-learn.jpg" style="height: 320px;"/></center>

> Hints: *Find possible insert, delete, & update anomalies*

<center><h2><strong><font color="blue"> Data Redundancy and Anomalies in Database </font></strong></h2></center>

Anomalies are problems that arise during the manipulation of a poorly structured table. In a relational database, **data redundancy** occurs when the same piece of data is stored in multiple places unnecessarily. While redundant backups are good, redundancy within a schema leads to significant operational risks known as **anomalies**.

### The Impact of Redundancy
* **Storage Inefficiency:** Large-scale research datasets (e.g., genomic data or longitudinal surveys) can become prohibitively expensive to store if duplicate information is recorded across millions of rows.
* **Data Inconsistency:** The primary danger is that one instance of a data point may be updated while others are not, leading to conflicting "truths" within the same dataset.

| Anomaly Type | Description | Research Example |
| :--- | :--- | :--- |
| **Update Anomaly** | Occurs when a change to a single data value requires multiple rows to be updated. | If a Lab Location changes, and it is stored in every row of an "Experiments" table, failing to update every row leads to inconsistent records. |
| **Insertion Anomaly** | Occurs when certain data cannot be recorded unless other, unrelated data is also provided. | You cannot record a new Researcher's profile until they are assigned to an active Experiment, because the primary key requires an Experiment ID. |
| **Deletion Anomaly** | Occurs when deleting one piece of information unintentionally results in the loss of other unrelated data. | Deleting the last Experiment entry for a specific Lab might accidentally delete all contact information for that Lab. |

<center><h2><strong><font color="blue"> Functional Dependencies (FD) </font></strong></h2></center>
**Functional Dependency** is a constraint between two sets of attributes in a relation. It is the fundamental building block of **Normalization**, the process used to eliminate redundancy and anomalies.

### Formal Definition
If $R$ is a relation and $X$ and $Y$ are subsets of attributes in $R$, we say $Y$ is **functionally dependent** on $X$ (denoted as $X \to Y$) if and only if each value of $X$ is associated with exactly one value of $Y$.

In simpler terms, if you know the value of $X$, you can determine the value of $Y$.
> **Example:** In a research database, `ResearcherID \to ResearcherName`. 
> If `ResearcherID` is 101, the name will always be "Dr. Smith." You won't find `ResearcherID` 101 associated with "Dr. Jones."

### Types of Functional Dependencies

To refine database structures, we identify specific categories of FDs:

#### A. Full Functional Dependency
An attribute is fully functionally dependent if it depends on the *entire* primary key, not just a part of it.
* **Example:** In a table where the key is `{Student_ID, Course_ID}`, the `Grade` is fully dependent because you need both the student and the course to determine the grade.

#### B. Partial Functional Dependency
This occurs when an attribute depends on only a portion of a composite primary key. This is a primary cause of redundancy.
* **Example:** In the same `{Student_ID, Course_ID}` table, `Student_Name` is only dependent on `Student_ID`. Storing it here creates partial dependency.

#### C. Transitive Dependency
This occurs when a non-key attribute depends on another non-key attribute, which in turn depends on the primary key ($A \to B$ and $B \to C$).
* **Example:** `Project_ID \to Department_ID` and `Department_ID \to Department_Head`. Here, `Department_Head` has a transitive dependency on `Project_ID`.

### Role in Normalization
Functional dependencies allow us to decompose a single, bloated table into smaller, logically sound tables:
1.  **First Normal Form (1NF):** Remove multi-valued attributes.
2.  **Second Normal Form (2NF):** Remove **Partial Dependencies**.
3.  **Third Normal Form (3NF):** Remove **Transitive Dependencies**.

By identifying $X \to Y$ relationships, researchers ensure that each fact is stored exactly once, preserving the integrity of the scientific record.

<center><img alt="" src="images/deds-05/functional-dependency.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue"> Phase 1: Achieving First Normal Form (1NF) </font></strong></h2></center>

## Conceptual Foundation
In the context of database normalization, **First Normal Form (1NF)** enforces *data atomicity* as a fundamental constraint. A relation is said to satisfy 1NF if it adheres to the following conditions:
1. **Atomic Attributes**  
   Each column must contain indivisible (atomic) values—no composite or multi-valued entries.
2. **No Repeating Groups**  
   Attributes must not store multiple values in a single cell (e.g., comma-separated lists).
3. **Row Uniqueness**  
   Every record must be uniquely identifiable via a **Primary Key**.

## Problem Identification
Consider the `online_retail_raw` table. Several records violate 1NF due to the presence of **multi-valued attributes**. For instance:
| order_id | product_id | product_name                  | unit_price     | quantity |
|----------|------------|-------------------------------|----------------|----------|
| 101      | 201, 205   | Wireless Mouse, Mechanical Keyboard | 25.00, 75.00 | 2, 1     |

### Observations
- Attributes such as `product_id`, `product_name`, `unit_price`, and `quantity` contain **comma-separated values**.
- This violates atomicity and introduces **repeating groups**.
- Consequently, standard SQL operations (e.g., filtering, aggregation, joins) become inefficient or infeasible.

## 1NF Normalization Procedure
### Step 1: Decompose Multi-Valued Attributes
Transform each multi-valued cell into multiple rows such that each row contains **exactly one value per attribute**.

#### Example Transformation (Order 101)
| order_id | product_id | product_name        | unit_price | quantity |
|----------|------------|---------------------|------------|----------|
| 101      | 201        | Wireless Mouse      | 25.00      | 2        |
| 101      | 205        | Mechanical Keyboard | 75.00      | 1        |

### Step 2: Enforce Uniform Row Structure
Apply the decomposition process across all records:
- Each product within an order becomes a separate row.
- Attributes such as `customer_name` and `customer_email` may repeat across rows—<font color="blue"><b>this is acceptable in 1NF</b></font>.
- The resulting table represents **transaction-level granularity**.

### Step 3: Define the Primary Key
Post-transformation:
- `order_id` is **not unique** (one order → multiple products).
- `product_id` is **not unique** (one product → multiple orders).

#### Solution: <font color="blue"><b>Composite Primary Key</b></font>
Define a **composite key**:
```sql
PRIMARY KEY (order_id, product_id)
````
This ensures that each row is uniquely identifiable.

## Result: 1NF-Compliant Table
* The original dataset (30 order records) is transformed into a larger set of **atomic transactional rows**.
* Each row now represents a single product within a specific order.
* The structure supports efficient querying, filtering, and aggregation.

<center><h1><strong><font color="blue"> Mini-Store First Normal Form (1NF) </font></strong></h1></center>

<center><img alt="" src="images/deds-06/1NF-Mini-Store.png" style="height: 600px;"/></center>

## Download the 1NF SQL Script ==> https://s.id/cdE3T

# Phase 2: Transitioning to Second Normal Form (2NF)
The progression to **Second Normal Form (2NF)** is essential for eliminating redundancy associated with partial functional dependencies. A relation is considered to be in 2NF if it satisfies two specific criteria:
1.  The table must already comply with **First Normal Form (1NF)**.
2.  Every non-prime attribute must be **fully functionally dependent** on the entire primary key.
In our current `online_retail_1nf` table, the composite primary key is defined as `{order_id, product_id}`. A violation occurs if a non-key attribute depends on only a portion of this composite key rather than the whole.

## 1. Identifying Partial Functional Dependencies
To normalize the dataset, we must evaluate the functional dependencies (FDs) of all non-key columns against the composite key `{order_id, product_id}`.

### Violation Group A: Order-Related Data
The following attributes are determined solely by the `order_id`. They do not require the `product_id` to be identified:
* `order_id` $\rightarrow$ `{order_date, customer_id, customer_name, customer_email, customer_country}`
* **Status:** Partial Dependency (Violation).

### Violation Group B: Product-Related Data
The following attributes are determined solely by the `product_id`. They do not vary based on the specific `order_id`:
* `product_id` $\rightarrow$ `{product_name, product_category, unit_price}`
* **Status:** Partial Dependency (Violation).

### Full Functional Dependency
Only the `quantity` attribute requires both the `order_id` and the `product_id` to determine its value, as it represents the specific count of a particular item within a specific order:
* `{order_id, product_id}` $\rightarrow$ `{quantity}`
* **Status:** Fully Functionally Dependent (Compliant).

<center><h1><strong><font color="blue"> Mini-Store Second Normal Form (2NF) </font></strong></h1></center>

<center><img alt="" src="images/deds-06/2nf-db-a.png" style="height: 600px;"/></center>

## 2. The Decomposition Process

To resolve these violations, the original table must be decomposed into three distinct relations. This ensures that every attribute in each table is fully dependent on its respective primary key.

### Step 1: Create the Orders Table
Extract all attributes related to the order and the customer. The `order_id` serves as the primary key.
* **Table:** `orders`
* **Columns:** `order_id` (PK), `order_date`, `customer_id`, `customer_name`, `customer_email`, `customer_country`.

### Step 2: Create the Products Table
Extract all attributes related to the inventory. The `product_id` serves as the primary key.
* **Table:** `products`
* **Columns:** `product_id` (PK), `product_name`, `product_category`, `unit_price`.

### Step 3: Create the Order Items Table
Maintain a linking table that captures the relationship between orders and products. This table retains the composite key and the only attribute that depends on both parts.
* **Table:** `order_items`
* **Columns:** `order_id` (FK), `product_id` (FK), `quantity`.

## Download the 1NF SQL Script ==> https://s.id/JQM9X

# Phase 3: Transitioning to Third Normal Form (3NF)

The transition to **Third Normal Form (3NF)** addresses the elimination of **transitive dependencies**. A database relation adheres to 3NF standards if it satisfies two essential conditions:
1.  The relation is already in **Second Normal Form (2NF)**.
2.  All non-prime attributes are non-transitively dependent on the primary key.

In our current `orders` table, while every attribute relates to the `order_id`, there is a hidden dependency chain where some information depends on the primary key only through another non-key attribute.

## 1. Identifying Transitive Functional Dependencies

In the 2NF `orders` table, we have the following structure:
* **Attributes:** `order_id` (PK), `order_date`, `customer_id`, `customer_name`, `customer_email`, `customer_country`.



### The Dependency Chain
We observe that `order_id` determines the `customer_id`. However, the `customer_id` subsequently determines the `customer_name`, `customer_email`, and `customer_country`. This creates a transitive dependency:
* `order_id` $\rightarrow$ `customer_id`
* `customer_id` $\rightarrow$ `{customer_name, customer_email, customer_country}`
* **Result:** `order_id` $\rightarrow$ `customer_id` $\rightarrow$ `{customer_name, customer_email, customer_country}`

### The Violation
In 3NF, "non-key attributes must provide a fact about the key, the whole key, and nothing but the key." The customer’s personal details are facts about the **Customer**, not the **Order**. If a customer places multiple orders, their name and email are redundantly stored in every order record, leading to potential data inconsistencies during updates.

<center><h1><strong><font color="blue"> Mini-Store Third Normal Form (3NF) </font></strong></h1></center>

<center><img alt="" src="images/deds-06/3NF-Mini-Store.png" style="height: 600px;"/></center>

## 2. The Decomposition Process
To achieve 3NF, we must remove the transitively dependent attributes and place them in a dedicated relation where the intermediate determinant becomes the Primary Key.

### Step 1: Create the Customers Table
Isolate all attributes that depend specifically on the `customer_id`.
* **Table:** `customers`
* **Columns:** `customer_id` (PK), `customer_name`, `customer_email`, `customer_country`.

### Step 2: Refine the Orders Table
Remove the customer details from the `orders` table, retaining only the `customer_id` as a **Foreign Key (FK)** to maintain the relationship.
* **Table:** `orders`
* **Columns:** `order_id` (PK), `order_date`, `customer_id` (FK).

## Download the 3NF SQL Script ==> https://s.id/ATAUL
## Delete 2NF order Table (skip foreign key check) before import the 3NF tables.

<center><h2><strong><font color="green">Tafakkur: Stop & Think</font></strong></h2></center>

<center><img alt="" src="images/curious.jpg" style="height: 320px;"/></center>

1. Consider the implications if normalization were performed only after the dataset had grown to encompass millions of transactional records.
2. Does the application of normalization forms facilitate the work of data scientists?
3. What other important information is not yet represented in our mini store example?

<center><img alt="" src="images/class-exercise.png" style="height: 600px;"/></center>

# Please form groups consisting of three to four members.

<center><h1><strong><font color="blue"> Case Study: Global Gym & Fitness Club </font></strong></h1></center>

## Background Information
The "Global Gym & Fitness Club" is a rapidly growing international fitness chain. To manage their operations, they currently use a single, large spreadsheet to track member registrations, class bookings, and instructor assignments. 

As the club expands to multiple countries, this "flat-file" approach has become a significant hurdle. The management is struggling with:
* **Data Entry Errors:** Inconsistent spelling of gym locations and instructor names.
* **Redundancy:** Every time a member joins a new class, their entire personal history is re-entered.
* **Inflexible Reporting:** It is difficult to calculate the total revenue per instructor or the popularity of specific class categories across different countries without complex manual filtering.

## The Learning Objective
In this exercise, students will act as Data Architects. Their task is to take this messy, unnormalized raw data and transform it into a structured, relational database. This transition will facilitate easier querying for business insights, such as:
1.  **Joins:** Connecting members to their specific classes and instructors.
2.  **Aggregations:** Calculating total members per country, average class prices, or total hours taught by each instructor.

<center><img alt="" src="images/deds-06/gym-db-casestudy.png" style="height: 600px;"/></center>

<center><h1><strong><font color="blue"> Case Study Data Sample: global_gym_db </font></strong></h1></center>

## Download here: https://s.id/SCzm9

<center><h1><strong><font color="blue"> global_gym_db: 1NF Solution? </font></strong></h1></center>

In [ ]:
# Solution here

<center><h1><strong><font color="blue"> global_gym_db: 2NF Solution? </font></strong></h1></center>

In [ ]:
# Solution here

<center><h1><strong><font color="blue"> global_gym_db: 3NF Solution? </font></strong></h1></center>

In [ ]:
# Solution here

In [2]:
# Python import the necessary modules
# Best Practice; use a function to connect to our database
import warnings; warnings.simplefilter('ignore')
import mysql.connector
import pandas as pd

host = "localhost"
user = "root"
password = ""
database = "" # modify as needed

def connect(host="localhost", user="root", password="", database=""):
    try:
        db = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database)
        con = db.cursor()
        print("Connected to the '{}' Database.".format(database))
        return db, con
    except Exception as err_:
        print("Connection to the Database server failed:")
        print(err_)

<center><h1><strong><font color="blue"> global_gym_db: Basic Query Exercise No 01 </font></strong></h1></center>

In [1]:
# Solution here

<center><h1><strong><font color="blue"> global_gym_db: Basic Query Exercise No 02 </font></strong></h1></center>


In [ ]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Basic Query Exercise No 03 </font></strong></h1></center>


In [3]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Join Query Exercise No 01 </font></strong></h1></center>


In [ ]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Join Query Exercise No 02 </font></strong></h1></center>


In [ ]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Join Query Exercise No 03 </font></strong></h1></center>

In [ ]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Aggregation Query Exercise No 01 </font></strong></h1></center>

In [ ]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Aggregation Query Exercise No 02 </font></strong></h1></center>

In [ ]:
#Solution here

<center><h1><strong><font color="blue"> global_gym_db: Aggregation Query Exercise No 03 </font></strong></h1></center>

In [ ]:
#Solution here

## Next Week's Discussions

* Entity-Relationship Diagrams (ERD).
* Data Acquisition II – Web Scraping & Crawling
   - Legal & Ethics of Scraping (robots.txt, Terms of Service).
   - Data Streaming
   - Normalization Assignment

<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-database-primary-keys.jpeg" style="height: 600px;"/></center>